<a href="https://colab.research.google.com/github/niranjansendilkumar11/Irish-weather-pipeline/blob/main/ca2_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import requests



In [ ]:
#API Configuration
from google.colab import userdata

try:
  API_KEY =userdata.get('OWM_API_KEY')
  print("API Key loaded successfully!")
except Exception:
  API_KEY = ""


API Key loaded successfully!


In [ ]:
IRISH_CITIES = [
    "Dublin,IE",
    "Cork,IE",
    "Galway,IE",
    "Limerick,IE",
    "Waterford,IE",
    "Swords,IE",
    "Sligo,IE",
    "Derry,IE",
    "Drogheda,IE",
    "Tallaght,IE"
]
print(f"Monitoring {len(IRISH_CITIES)} Irish cities:")
for city in IRISH_CITIES:
    print(f"{city.split(',')[0]}")


Monitoring 10 Irish cities:
Dublin
Cork
Galway
Limerick
Waterford
Swords
Sligo
Derry
Drogheda
Tallaght


In [ ]:
import time

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

def fetch_city_weather(city_name):
    params = {
        "q": city_name,
        "appid": API_KEY,
        "units": "metric"
    }
    try:
      response = requests.get(BASE_URL, params=params,timeout=10)
      if response.status_code == 200:
        data = response.json()
        return {
                "city":          data["name"],
                "country":       data["sys"]["country"],
                "latitude":      data["coord"]["lat"],
                "longitude":     data["coord"]["lon"],
                "temp_celsius":  data["main"]["temp"],
                "feels_like":    data["main"]["feels_like"],
                "temp_min":      data["main"]["temp_min"],
                "temp_max":      data["main"]["temp_max"],
                "humidity":      data["main"]["humidity"],
                "pressure":      data["main"]["pressure"],
                "visibility":    data.get("visibility", None),
                "wind_speed":    data["wind"]["speed"],
                "wind_degree":   data["wind"].get("deg", None),
                "weather_main":  data["weather"][0]["main"],
                "weather_desc":  data["weather"][0]["description"],
                "cloud_coverage":data["clouds"]["all"],
                "sunrise":       data["sys"]["sunrise"],
                "sunset":        data["sys"]["sunset"],
                "fetched_at":    datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            }
      else:
            print(f"Failed for {city_name} - Status: {response.status_code}")
            return None
    except requests.exceptions.Timeout:
        print(f"Timeout for {city_name}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Error for {city_name}: {e}")
        return None


In [ ]:
result = fetch_city_weather ("Dublin,IE")
print(result)

{'city': 'Dublin', 'country': 'IE', 'latitude': 53.344, 'longitude': -6.2672, 'temp_celsius': 5.86, 'feels_like': 1.55, 'temp_min': 4.53, 'temp_max': 6.81, 'humidity': 75, 'pressure': 1019, 'visibility': 10000, 'wind_speed': 7.2, 'wind_degree': 290, 'weather_main': 'Clouds', 'weather_desc': 'scattered clouds', 'cloud_coverage': 40, 'sunrise': 1774419349, 'sunset': 1774464372, 'fetched_at': '2026-03-25 19:32:54'}


In [ ]:
#Section 4 - Data Acquisition
# This section fetches live weather data for all 10 Irish cities
def fetch_all_cities(cities):
  all_records = []
  for city in cities:
    print(f"Fetching {city.split(',')[0]}...")
    record = fetch_city_weather(city)
    if record:
      all_records.append(record)
    time.sleep(1)
  df_raw = pd.DataFrame(all_records)
  print(f"Done Fetched {len(all_records)} cities")
  return df_raw

df_raw = fetch_all_cities(IRISH_CITIES)


Fetching Dublin...
Fetching Cork...
Fetching Galway...
Fetching Limerick...
Fetching Waterford...
Fetching Swords...
Fetching Sligo...
Fetching Derry...
Fetching Drogheda...
Fetching Tallaght...
Done Fetched 10 cities


In [ ]:
# Displaying the raw  data we fetched
df_raw.head(10)

,city,country,latitude,longitude,temp_celsius,feels_like,temp_min,temp_max,humidity,pressure,visibility,wind_speed,wind_degree,weather_main,weather_desc,cloud_coverage,sunrise,sunset,fetched_at
0,Dublin,IE,53.3440,-6.2672,5.86,1.55,4.53,6.81,75,1019,10000,7.20,290,Clouds,scattered clouds,40,1774419349,1774464372,2026-03-25 19:32:54
1,Cork,IE,51.8980,-8.4706,6.87,3.84,5.40,6.87,70,1024,10000,4.63,310,Clouds,broken clouds,75,1774419917,1774464861,2026-03-25 19:32:55
2,Galway,IE,53.2719,-9.0489,6.35,2.55,6.35,6.35,65,1023,10000,6.15,299,Clouds,overcast clouds,94,1774420018,1774465039,2026-03-25 19:32:56
3,Limerick,IE,52.6647,-8.6231,6.97,3.00,6.97,6.97,68,1023,10000,7.13,309,Clouds,scattered clouds,38,1774419933,1774464919,2026-03-25 19:32:57
4,Waterford,IE,52.2583,-7.1119,6.50,3.38,6.10,7.27,70,1022,10000,4.63,310,Clouds,scattered clouds,40,1774419582,1774464545,2026-03-25 19:32:58
5,Swords,IE,53.4597,-6.2181,5.48,1.06,4.47,6.75,74,1019,10000,7.20,290,Clouds,scattered clouds,40,1774419334,1774464364,2026-03-25 19:32:59
6,Sligo,IE,54.2697,-8.4694,6.00,0.78,6.00,6.00,72,1021,10000,10.53,310,Clouds,overcast clouds,86,1774419849,1774464929,2026-03-25 19:33:00
7,Derry,IE,51.5867,-9.0503,5.90,1.58,5.90,5.90,69,1025,10000,7.25,312,Clouds,scattered clouds,48,1774420065,1774464992,2026-03-25 19:33:01
8,Drogheda,IE,53.7189,-6.3478,5.63,1.24,5.63,5.63,71,1019,10000,7.24,304,Rain,moderate rain,24,1774419357,1774464403,2026-03-25 19:33:02
9,Tallaght,IE,53.2859,-6.3734,4.91,0.33,3.94,6.23,77,1020,10000,7.20,290,Clouds,few clouds,20,1774419376,1774464396,2026-03-25 19:33:03


In [ ]:
#section 5- Transformations
# This section cleans and enriches the raw weather data

def categorise_temperature (temp_celsius):
  if temp_celsius < 0:
    return "Freezing"
  elif temp_celsius < 8:
        return "Cold"
  elif temp_celsius < 14:
        return "Cool"
  elif temp_celsius < 19:
        return "Mild"
  else:
        return "Warm"

In [ ]:
def categorise_wind(wind_speed):
    if wind_speed < 3.0:
        return "Calm"
    elif wind_speed < 8.0:
        return "Breezy"
    elif wind_speed < 14.0:
        return "Windy"
    else:
        return "Storm"


In [ ]:
def categorise_humidity(humidity):
  if humidity < 40:
        return "Dry"
  elif humidity < 70:
        return "Comfortable"
  else:
        return "Humid"

In [ ]:
#calculating how many hours of daylight each city gets
def convert_unix_timestamp(unix_time):
    return datetime.fromtimestamp(unix_time).strftime('%H:%M:%S')

def calculate_daylight_hours(sunrise_unix, sunset_unix):
    duration_seconds = sunset_unix - sunrise_unix
    return round(duration_seconds / 3600, 2)

print(convert_unix_timestamp(1774333095))
print(calculate_daylight_hours(1774333095, 1774377864))

06:18:15
12.44


In [ ]:
# Applies all transformations to the raw dataframe and returns a cleaned enriched version
def transform_weather_data(df):
  df_transformed = df.copy()

  df_transformed["sunrise_time"] = df_transformed["sunrise"].apply(convert_unix_timestamp)
  df_transformed["sunset_time"] = df_transformed["sunset"].apply(convert_unix_timestamp)
  df_transformed["daylight_hours"] = df_transformed.apply(
        lambda row: calculate_daylight_hours(row["sunrise"], row["sunset"]), axis=1
    )
  df_transformed["temp_category"] = df_transformed["temp_celsius"].apply(categorise_temperature)
  df_transformed["wind_category"] = df_transformed["wind_speed"].apply(categorise_wind)
  df_transformed["humidity_category"] = df_transformed["humidity"].apply(categorise_humidity)
  df_transformed["visibility_km"] = (df_transformed["visibility"] / 1000).round(2)
  df_transformed.drop(columns=["sunrise", "sunset", "visibility"], inplace=True)

  return df_transformed

df_transformed = transform_weather_data(df_raw)
print("Transformations applied successfully!")
print(f"Columns now: {df_transformed.shape[1]}")
df_transformed.head(10)

Transformations applied successfully!
Columns now: 23


,city,country,latitude,longitude,temp_celsius,feels_like,temp_min,temp_max,humidity,pressure,...,weather_desc,cloud_coverage,fetched_at,sunrise_time,sunset_time,daylight_hours,temp_category,wind_category,humidity_category,visibility_km
0,Dublin,IE,53.3440,-6.2672,5.86,1.55,4.53,6.81,75,1019,...,scattered clouds,40,2026-03-25 19:32:54,06:15:49,18:46:12,12.51,Cold,Breezy,Humid,10.0
1,Cork,IE,51.8980,-8.4706,6.87,3.84,5.40,6.87,70,1024,...,broken clouds,75,2026-03-25 19:32:55,06:25:17,18:54:21,12.48,Cold,Breezy,Humid,10.0
2,Galway,IE,53.2719,-9.0489,6.35,2.55,6.35,6.35,65,1023,...,overcast clouds,94,2026-03-25 19:32:56,06:26:58,18:57:19,12.51,Cold,Breezy,Comfortable,10.0
3,Limerick,IE,52.6647,-8.6231,6.97,3.00,6.97,6.97,68,1023,...,scattered clouds,38,2026-03-25 19:32:57,06:25:33,18:55:19,12.50,Cold,Breezy,Comfortable,10.0
4,Waterford,IE,52.2583,-7.1119,6.50,3.38,6.10,7.27,70,1022,...,scattered clouds,40,2026-03-25 19:32:58,06:19:42,18:49:05,12.49,Cold,Breezy,Humid,10.0
5,Swords,IE,53.4597,-6.2181,5.48,1.06,4.47,6.75,74,1019,...,scattered clouds,40,2026-03-25 19:32:59,06:15:34,18:46:04,12.51,Cold,Breezy,Humid,10.0
6,Sligo,IE,54.2697,-8.4694,6.00,0.78,6.00,6.00,72,1021,...,overcast clouds,86,2026-03-25 19:33:00,06:24:09,18:55:29,12.52,Cold,Windy,Humid,10.0
7,Derry,IE,51.5867,-9.0503,5.90,1.58,5.90,5.90,69,1025,...,scattered clouds,48,2026-03-25 19:33:01,06:27:45,18:56:32,12.48,Cold,Breezy,Comfortable,10.0
8,Drogheda,IE,53.7189,-6.3478,5.63,1.24,5.63,5.63,71,1019,...,moderate rain,24,2026-03-25 19:33:02,06:15:57,18:46:43,12.51,Cold,Breezy,Humid,10.0
9,Tallaght,IE,53.2859,-6.3734,4.91,0.33,3.94,6.23,77,1020,...,few clouds,20,2026-03-25 19:33:03,06:16:16,18:46:36,12.51,Cold,Breezy,Humid,10.0
